# 38 — gen-3.6 screen · LANZAR EN EL POD

**Handoff 2026-08-12 (legokna → quien lance), medido en el pod el 13-ago.** Todo está construido y
validado; falta correr.

**Corre en el POD**, no en UNAM: necesita los datos del challenge, que no pueden estar en la
máquina universitaria hasta que haya acuerdo escrito (`context/UNAM_SERVER.md`).

## Qué está ya validado (en UNAM, cero datos del challenge, cero GPU de entrenamiento)

| | dónde |
|---|---|
| Tubería cargar→LoRA→2 pasos→merge, 6/6 | `RESULTS_smoke_unsloth.json` |
| La ruta de eval lee un merge de Unsloth | `RESULTS_eval_path.json` |
| La ruta de entrega FP8 | `RESULTS_fp8_llmcompressor.json` |
| Por qué NO es ms-swift | `RESULTS_viability_v2.json` |

## Ya medido en el pod con datos reales (`RESULTS_smoke_pod_27b.json`)

- ✅ **23,5 s/it sostenido** (el paso 1 fue 230,6 s: compilación y warm-up) × 900,9 pasos ⇒
  **una época ≈ 5,9 h**. ⚠️ Es UNA medición sostenida; trátala como indicativa.
- 🔴 **Pico 52,64 GiB ⇒ hace falta una GPU de ≥80 GB.** No cabe en un L40S de 48.
- 🔴 **`df` MIENTE en los volúmenes de RunPod** — reporta el clúster MooseFS, no la cuota (~640 GB).
  Comprueba el espacio de verdad; el 13-ago un merge murió por `Disk quota exceeded`.

## El sujeto: **`Qwen/Qwen3.6-27B`** — decidido 2026-08-13

Necesita el paso FP8 para servir (celda 5), ya validado. La alternativa medida y descartada era
`Qwen/Qwen3.5-9B` (~18 GB bf16, sin paso FP8, lectura más limpia por ser casi paritario con
nuestro 8B). Cambiar de uno a otro sigue siendo **una línea** en la celda 4.

## 1 · Entorno — usar la ruta DOCUMENTADA de Unsloth

⚠️ En UNAM usamos conda + pip plano y nos costó un `torchvision::nms` roto. Su doc dice
`uv pip install unsloth --torch-backend=auto` **en venv**, y *"do NOT use this if you have Conda"*.
En el pod, seguir su ruta.

In [ ]:
!python -m venv /workspace/envs/unsloth && \
 /workspace/envs/unsloth/bin/pip install -q uv && \
 /workspace/envs/unsloth/bin/uv pip install --python /workspace/envs/unsloth/bin/python \
     unsloth --torch-backend=auto

# verificación: torch, GPU y que unsloth importe
!/workspace/envs/unsloth/bin/python -c "\
import unsloth, torch, transformers; \
print('unsloth', unsloth.__version__, '| transformers', transformers.__version__); \
print('torch', torch.__version__, '| gpus', torch.cuda.device_count(), \
      '| cap', torch.cuda.get_device_capability(0)); \
from unsloth import FastVisionModel; print('FastVisionModel OK')"

## 2 · Comprobar rutas antes de nada

La guarda de sha256 caza un `train.jsonl` equivocado al instante, pero mejor verlo antes.

In [ ]:
from pathlib import Path
import hashlib

TRAIN = Path('/workspace/repo/experiments/18-count-aug/runs/18_count_aug_v1/train.jsonl')
print('existe:', TRAIN.exists())
if TRAIN.exists():
    print('filas :', sum(1 for _ in open(TRAIN)))
    print('sha256:', hashlib.sha256(TRAIN.read_bytes()).hexdigest())
    print('esperado: 180e28f0325674197d52706beeabd846851bdd2875264b5c3505e0debfbd8e8b')
    import json
    r = json.loads(open(TRAIN).readline())
    img = Path(r['images'][0])
    print('primer frame existe:', img.exists(), '→', img)

## 3 · SMOKE con datos reales — 2 pasos

🔴 **No saltar.** Confirma que el 27B carga, que el `train.jsonl` real convierte, y **da el `s/it`**.

In [ ]:
import sys; sys.path.insert(0, '/workspace/repo/experiments/38-gen36-ft-screen')
from _models.unsloth_sft import Config, main

res = main(Config(run_name='38_smoke', smoke=True, smoke_steps=2, smoke_rows=32))
print('\nLoRA por parte:', res['lora_by_part'])   # merger 0 es ESPERADO, no un fallo
print('pico VRAM      :', res['peak_vram_gib'], 'GiB')
print('segundos       :', res['train_secs'], '→ mira el s/it antes de la celda 4')

## 4 · EL BRAZO — 1 época

Lánzalo dentro de **`tmux`** (`tmux new -s ft38`) para que una desconexión no lo mate.

**Si muere, se reanuda: vuelve a correr ESTA MISMA celda, sin cambiar nada.** `main()` busca el
último checkpoint en `runs/38_qwen36_27b_v1/ckpt/` y continúa desde su paso; si no hay ninguno,
arranca limpio. `RESULTS_train.json` registra `resumed_from`, así que un número que salga de un run
reanudado se puede identificar como tal.

- Guarda cada **100 pasos** (~39 min de los 900,9 pasos), conservando los 2 últimos.
- **Dónde va sin abrir nada:** `cat runs/38_qwen36_27b_v1/HEARTBEAT.json` → paso, %, loss, s/it y
  ETA en horas. El log completo queda en `runs/38_qwen36_27b_v1/train.log` (fichero, no scrollback).

🔴 **La GPU:** pico medido **52,64 GiB** ⇒ hace falta **≥80 GB** (A100/H100). No cabe en un L40S 48.

In [ ]:
cfg = Config(
    model='Qwen/Qwen3.6-27B',      # ← alternativa: 'Qwen/Qwen3.5-9B' (sin FP8 al servir)
    run_name='38_qwen36_27b_v1',
    num_train_epochs=1,
    # receta A2 trasladada; los defaults ya la llevan
    learning_rate=2e-4, lora_rank=8, lora_alpha=32,
    gradient_accumulation_steps=16, seed=42,
    max_pixels=1280*720,           # 🔴 Unsloth por defecto usa 512
)
res = main(cfg)
res

## 5 · Cuantizar a FP8 (sólo si el sujeto es el 27B)

**Data-free**: no necesita datos, así que puede correr donde sea.
⚠️ **Entorno SEPARADO** — `llmcompressor` pide `transformers>=5.9.0` y Unsloth capa en `<=5.5.0`.

In [ ]:
!python -m venv /workspace/envs/quant && /workspace/envs/quant/bin/pip install -q llmcompressor
!/workspace/envs/quant/bin/python /workspace/repo/experiments/38-gen36-ft-screen/_tools/quantize_fp8.py \
    /workspace/repo/experiments/38-gen36-ft-screen/runs/38_qwen36_27b_v1/merged \
    /workspace/repo/experiments/38-gen36-ft-screen/runs/38_qwen36_27b_v1/merged_fp8

## 6 · Evaluar y comparar

### ✅ LA VARA — RESUELTA 2026-08-13 (sustituye a la nota "en disputa")

Los dos números **son reales y salen de `frame.metrics`**. Lo que había no era una cifra inventada
sino **un conflicto entre dos documentos de este mismo rung**.

| documento | qué pone |
|---|---|
| Esta celda, versión original | *"La vara, escrita antes de ver números — batir a **A2 ep3**: proxy local 0.6104, `bucket_mean` 0.6496"* |
| README, §"Why this rung exists" | el test es *una época leída contra el checkpoint de **época 1** de A2* |

La celda dice ep3 **a propósito**, no por copiar mal una fila. Y el `0.5282` que se propuso el
13-ago como alternativa tampoco servía: sale de `02-lora-sft/` — el rung **02**, otro brazo.

**Fuente primaria: `experiments/21-recipe-sweep/RESULTS_A2_lr.csv`** — rung **21**, brazo `A2_lr`
(`21_lr_2e4_v1`, lr 2e-4, r8/α32), que es el A2 que este rung traslada. Cada fila salió de
`21b_epoch_eval.ipynb`: merge del checkpoint de esa época → inferencia sobre las 6.252 preguntas
del test → juez LLM → `stratified_report`. Son mediciones, no transcripciones.

| epoch | checkpoint | `proxy_leaderboard` | `bucket_mean` |
|---|---|---|---|
| **1** | **`checkpoint-901`** | **0.4986** | **0.5592** |
| 2 | `checkpoint-1802` | 0.5751 | 0.6153 |
| 3 | `checkpoint-2703` | 0.6104 | 0.6496 |

`0.6104` y `0.6496` son el proxy y el `bucket_mean` de **la misma fila: A2 ep3**. `0.6496` **no** es
un `ci_low`, como se sospechó.

### ⇒ GANA EL README: la vara es A2 **ep1**

> **A2 ep1 · `checkpoint-901` · proxy `0.4986` · `bucket_mean` `0.5592`**

Tres razones, en orden de peso:

1. **Emparejamiento por época — y aquí por paso.** Nuestro brazo corrió **901 pasos**; el checkpoint
   de A2 ep1 es **`checkpoint-901`**. Coinciden exactamente.
2. **Este error ya nos costó dos veces.** `21b_epoch_eval.ipynb` lo dice de sí mismo: *"los rungs 14
   y 15 compararon ambos su época 3 contra la época 2 del rung 06, y uno de los dos registró una
   victoria que vive enteramente en ese desajuste"*. Es el fallo que el emparejamiento existe para
   impedir.
3. **ep1 no es "convergido", es el schedule a medias.** El coseno recoce a lr 0 en el último paso
   planificado, así que ningún brazo del rung 21 dejó de subir en ep3. Pedirle a un 27B de 1 época
   que bata a un 8B de 3 cierra la vía gen-3.6 sobre una comparación que no mide lo que dice medir.

🔴 Lo que **sí** conserva sentido de la vara original: si el brazo va a la entrega, tendrá que batir
a A2 ep3 (`0.6104` / `0.6496`) — pero eso se decide **con 3 épocas entrenadas**, no aquí.

### Cómo se corre

Las celdas de abajo replican `21b_epoch_eval.ipynb`, con dos desviaciones deliberadas:
**no hay paso de merge** (los 52 GB ya están) y **`engine_factory = GenericVLMEngine`**
(`Qwen3_5ForConditionalGeneration` no carga bajo el pin de transformers 4.57).

1. `SMOKE = True` primero — 40 preguntas, prueba el cableado entero. Barato.
2. Luego `SMOKE = False` — las 6.252. **Dentro de `tmux`.**

**Sólo hay UNA inferencia en GPU: la nuestra.** El control se re-puntúa desde las respuestas ya
archivadas de A2 ep1 (`…/21_lr_2e4_v1/ep1_full/results.csv`, verificado en el pod, 6.252 filas):
`metrics.py:14` — *"Pure pandas/numpy … no torch, no GPU, no judge"*. Segundos de CPU.

Se recalcula en vez de citar el `0.4986` porque un número transcrito no se puede re-derivar — y eso
da un test gratis: **si el control no reproduce `0.4986` / `0.5592`, el roto es nuestro eval, no la
vara, y hay que parar antes de leer nada del 27B.**

⚠️ Aun así el resultado es un **suelo**: el README (`:129-130`) avisa de que la receta A2 no es
known-good en otro backbone; `lr 2e-4` es el óptimo hallado *para el 8B*.

⚠️ Deflactar antes de leerlo como leaderboard: el `bucket_mean` local sobreestima **+0.121** y el
orden de buckets está invertido en `obj_OOD` ([[local-eval-vs-judge-calibration]]). El signo
sobrevive 8/8; la magnitud no.

In [ ]:
# --- bootstrap + parámetros + PRE-FLIGHT del juez -------------------------------
# Modelado sobre experiments/21-recipe-sweep/21b_epoch_eval.ipynb, que es el código que
# produjo los números de A2. Dos diferencias deliberadas:
#   1. NO hay paso de merge — el merge de 52 GB ya existe y se apunta directamente.
#   2. `engine_factory` = GenericVLMEngine: Qwen3_5ForConditionalGeneration no carga bajo
#      transformers 4.57 (AutoConfig levanta KeyError: 'qwen3_5').
import json, logging, os, sys, time
from pathlib import Path
import pandas as pd

EXP = Path('/workspace/repo_leo/experiments/38-gen36-ft-screen')
REPO = EXP.parent.parent
for p in (REPO / 'src', REPO / 'vendor' / 'orena-focus' / 'src',
          REPO / 'experiments' / '23-backbone-screen' / '_tools'):
    if p.is_dir():
        sys.path.insert(0, str(p))

# 🔴 HF_HOME ANTES que los flags offline, y RESUELTO MIRANDO, no a pelo.
# Este pod tiene DOS cachés de HF y el juez sólo está en una:
#   /workspace/hf_cache            -> el 27B y Qwen3.5-4B. De Qwen3-4B sólo tiene .locks
#                                     (una descarga abortada) => LocalEntryNotFoundError.
#   /workspace/.cache/huggingface  -> el juez Qwen3-4B completo, 7,6 GB, 3 shards.
# El notebook del rung 21 fijaba /workspace/hf_cache, pero eso era OTRO pod. Fijarlo a pelo
# aquí revienta el gate; peor, sin gate reventaría después de la pasada de inferencia entera.
_JUDGE_REPO_DIR = 'models--Qwen--Qwen3-4B'      # BaselineConfig().judge_model == 'Qwen/Qwen3-4B'
_HF_CANDIDATES = [Path('/workspace/.cache/huggingface'), Path('/workspace/hf_cache')]
_hf_home = next(
    (c for c in _HF_CANDIDATES
     if any((c / 'hub' / _JUDGE_REPO_DIR / 'snapshots').glob('*/config.json'))), None)
assert _hf_home is not None, (
    f'el juez ({_JUDGE_REPO_DIR}) no está completo en ninguna de {_HF_CANDIDATES}. '
    'Un .locks/ suelto NO cuenta: es una descarga abortada.')
os.environ['HF_HOME'] = str(_hf_home)
os.environ.setdefault('HF_HUB_OFFLINE', '1')       # seguro aquí: el merge YA está hecho
os.environ.setdefault('TRANSFORMERS_OFFLINE', '1')
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(name)s %(message)s',
                    datefmt='%H:%M:%S')

from frame import ledger, metrics
from frame.config import BaselineConfig
from frame.run import run_baseline
from screen_engine import GenericVLMEngine

# --- parámetros -----------------------------------------------------------------
SMOKE  = True      # True -> 40 preguntas, sólo cableado. Completo: SMOKE = False
RUN    = '38_qwen36_27b_v1'
MERGED = EXP / 'runs' / RUN / 'merged'
DATA_ROOT = Path('/workspace/orena-data')

# 🔴 EL CONTROL: A2 época 1 = checkpoint-901, y nuestro brazo corrió 901 pasos.
# Emparejado por época Y por paso. Se RECALCULA desde sus respuestas archivadas
# (RULES §6b), nunca se transcribe de RESULTS_A2_lr.csv.
CTRL_DIR = Path('/workspace/repo/experiments/21-recipe-sweep/runs/21_lr_2e4_v1/ep1_full')

RUN_DIR = EXP / 'runs' / RUN
RUN_TAG = 'ep1_smoke' if SMOKE else 'ep1_full'

assert MERGED.is_dir() and any(MERGED.iterdir()), f'no hay merge en {MERGED}'
assert (DATA_ROOT / 'heico/data/frame/test.parquet').exists(), 'sin test.parquet'
assert (CTRL_DIR / 'results.csv').exists(), (
    f'{CTRL_DIR}/results.csv falta — sin las respuestas de A2 ep1 no hay control que '
    'recalcular, y un control transcrito de una tabla no es un control (RULES §6b)')

# --- PRE-FLIGHT: el juez debe resolver offline ANTES de gastar nada --------------
# run_baseline carga el juez sólo DESPUÉS de la pasada de inferencia entera, así que una
# caché mal resuelta falla con todo ya pagado. Esto es ese fallo, adelantado y barato:
# el 13-ago cazó exactamente este HF_HOME equivocado en 40 segundos.
from transformers import AutoTokenizer
_judge = BaselineConfig().judge_model
try:
    AutoTokenizer.from_pretrained(_judge)
except Exception as exc:
    raise AssertionError(
        f'GATE DEL JUEZ: {_judge!r} no resuelve offline ({type(exc).__name__}). '
        f'HF_HOME={os.environ.get("HF_HOME")!r}. Arregla el entorno — NO quites los flags '
        'offline, la historia de despliegue entera es offline.') from exc

print(f'OK juez  : {_judge} resuelve desde {os.environ.get("HF_HOME")}')
print(f'OK merge : {MERGED}')
print(f'OK control: {CTRL_DIR} ({sum(1 for _ in open(CTRL_DIR / "results.csv")) - 1} filas)')
print(f'MODO     : {"SMOKE (40 preguntas)" if SMOKE else "COMPLETO (6252 preguntas)"}')

In [ ]:
# --- EL EVAL --------------------------------------------------------------------
# Lánzalo dentro de tmux: la pasada completa son 6252 preguntas sobre un 27B.
t0 = time.perf_counter()
cfg_eval = BaselineConfig(
    data_root=DATA_ROOT, model_path=MERGED, out_dir=RUN_DIR, run_name=RUN_TAG,
    max_pixels=1280 * 720,     # el mismo que entrenó el brazo Y el que evaluó a A2
    seed=42, n_eval=40 if SMOKE else None,
)
# El backbone es el sujeto de este rung; la ruta de inferencia NO puede ser una segunda
# variable. Estas cuatro son las mismas asserts que corrió el eval de A2.
assert cfg_eval.answer_postprocess is None, 'answer_postprocess debe seguir None'
assert cfg_eval.n_samples == 1 and cfg_eval.enhance is None and cfg_eval.aux_view is None

# 🔴 La ÚNICA desviación respecto del eval de A2, y es forzada: Qwen3_5ForConditionalGeneration
# no carga bajo el pin de transformers 4.57. GenericVLMEngine mantiene idénticos el
# SYSTEM_PROMPT (importado, no copiado), la forma del mensaje, greedy, max_new_tokens y
# answer_char_cap — y suprime el CoT, que si no se come el presupuesto de tokens.
cfg_eval.engine_factory = GenericVLMEngine

report = run_baseline(cfg_eval)
print(f'\neval terminado en {(time.perf_counter() - t0) / 60:.1f} min')

In [ ]:
# --- gates canónicos + LA COMPARACIÓN vs A2 ep1 ---------------------------------
# Puntuar SÓLO por frame.metrics, nunca re-derivar a mano (RULES: EVAL).
gold = ledger.gold_from_frame_parquets(DATA_ROOT)
res  = pd.read_csv(RUN_DIR / RUN_TAG / 'results.csv')

missing = set(res['qID']) - set(gold.dropna(subset=['answer'])['qID'])
assert not missing, f'GATE 0 — {len(missing)} qIDs sin gold; todo margen saldría inflado'
metrics.assert_no_dup_qid(res)
metrics.assert_ood_from_qid(res)
metrics.assert_all_rows_grouped(res)
strat = metrics.stratified_report(res, gold=gold)
metrics.assert_floors_vs_eval_set(strat)

# Gate de modo sobre el ARTEFACTO: un SMOKE=False que no surtió efecto es indistinguible
# de un run completo con éxito si sólo se mira la variable.
assert (len(res) < 1000) == SMOKE, f'GATE DE MODO: SMOKE={SMOKE} pero se puntuaron {len(res)} filas'
if not SMOKE:
    assert len(res) == 6252, f'se esperaba el eval completo, salieron {len(res)} filas'

# --- el proxy de leaderboard = mean(aggregation_ID, object_recognition_ID) -------
# NO es bucket_mean; son cantidades distintas y no se citan una contra otra (RULES §4b).
def _proxy(report: dict) -> dict:
    bb  = pd.DataFrame(report['by_bucket'])
    idc = bb[bb.distribution == 'ID'].set_index('capability_group')
    out = {}
    for key, name in (('aggregation_ID', 'aggregation'),
                      ('object_recognition_ID', 'object_recognition')):
        out[key] = float(idc.loc[name, 'accuracy']) if name in idc.index else float('nan')
    out['proxy'] = (out['aggregation_ID'] + out['object_recognition_ID']) / 2
    return out

arm_p = _proxy(strat)
if not SMOKE and any(pd.isna(v) for v in arm_p.values()):
    raise AssertionError(
        f'falta un bucket puntuado en el eval completo: {arm_p} — el proxy es la media de '
        'exactamente esos dos, así que uno ausente no es un número más pequeño, es ningún número')

# 🔴 EL CONTROL, RECALCULADO desde las respuestas archivadas de A2 ep1 por ESTE mismo código.
# Si esto se transcribiera de RESULTS_A2_lr.csv no sería re-derivable — y ese fue exactamente
# el fallo que bloqueó este rung el 13-ago.
ctrl_res   = pd.read_csv(CTRL_DIR / 'results.csv')
ctrl_strat = metrics.stratified_report(ctrl_res, gold=gold)
ctrl_p     = _proxy(ctrl_strat)

print('\n' + '=' * 66)
print('  27B gen-3.6 ep1 (901 pasos)  vs  A2 ep1 (checkpoint-901)')
print('=' * 66)
cmp_df = pd.DataFrame([
    {'metrica': 'proxy_leaderboard',     'A2_ep1': ctrl_p['proxy'],                 'arm_27b': arm_p['proxy']},
    {'metrica': 'aggregation_ID',        'A2_ep1': ctrl_p['aggregation_ID'],        'arm_27b': arm_p['aggregation_ID']},
    {'metrica': 'object_recognition_ID', 'A2_ep1': ctrl_p['object_recognition_ID'], 'arm_27b': arm_p['object_recognition_ID']},
    {'metrica': 'bucket_mean',           'A2_ep1': ctrl_strat['bucket_mean'],       'arm_27b': strat['bucket_mean']},
    {'metrica': 'margin_OOD',            'A2_ep1': ctrl_strat['margin_OOD'],        'arm_27b': strat['margin_OOD']},
    {'metrica': 'acc_ID',                'A2_ep1': ctrl_strat['acc_ID'],            'arm_27b': strat['acc_ID']},
    {'metrica': 'acc_OOD',               'A2_ep1': ctrl_strat['acc_OOD'],           'arm_27b': strat['acc_OOD']},
])
cmp_df['delta'] = cmp_df['arm_27b'] - cmp_df['A2_ep1']
print(cmp_df.round(4).to_string(index=False))

# 🔴 Sanidad: el control recalculado tiene que reproducir la tabla publicada del rung 21.
# Si no cuadra, el que está mal es el eval, no la vara — y hay que parar.
print(f'\ncontrol recalculado : proxy {ctrl_p["proxy"]:.4f} · bucket_mean {ctrl_strat["bucket_mean"]:.4f}')
print( 'RESULTS_A2_lr.csv   : proxy 0.4986 · bucket_mean 0.5592')

# --- el veredicto pre-registrado ------------------------------------------------
d_proxy = arm_p['proxy'] - ctrl_p['proxy']
d_ood   = strat['margin_OOD'] - ctrl_strat['margin_OOD']
print(f'\nLECTURA PRE-REGISTRADA (vs A2 época 1)')
print(f'   el proxy sube      : {d_proxy > 0}  ({d_proxy:+.4f})')
print(f'   margin_OOD no cae  : {d_ood >= 0}  ({d_ood:+.4f})')
print(f'   -> {"GO" if (d_proxy > 0 and d_ood >= 0) else "NO ES UNA VICTORIA"}')
print('\n⚠️  Es un SUELO, no un techo: la receta A2 (lr 2e-4) es el óptimo hallado para el 8B,')
print('    no está portada a este backbone (README :129-130). Y el bucket_mean local')
print('    sobreestima +0.121 con obj_OOD invertido — usa el signo, no la magnitud.')
print('⚠️  Sin estimación de varianza por semilla: ninguna semilla se ha repetido nunca.')